In [1]:
import sys
import os

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)


from vllm_wrapper import VLLMRaterModel
from eval.safe.rate_atomic_fact import check_atomic_fact

ModuleNotFoundError: No module named 'pyserini'

In [2]:
import json
atomic_fact_data = []
# In Jupyter notebooks, use getcwd() instead of __file__
path = os.path.join(os.getcwd(), "data_for_git", "atomic_facts.jsonl")
with open(path, "r") as f:
    for line in f:
        atomic_fact_data.append(json.loads(line))

In [14]:
llm = VLLMRaterModel()
import threading
from tqdm import tqdm

max_concurrent_requests = 50

request_semaphore = threading.Semaphore(max_concurrent_requests)

# Thread-safe list to collect results
results = []
results_lock = threading.Lock()
error_log = []
error_log_lock = threading.Lock()
pbar_lock = threading.Lock()
def worker(full_dict, llm, pbar):
    try:
        request_semaphore.acquire()
        bio_person = full_dict["prompt"].split("Tell me a bio of ")[1]
        all_atomic_facts = full_dict["results"]["all_atomic_facts"]
        for sentence_facts in all_atomic_facts:
            for fact_index, fact in enumerate(sentence_facts["atomic_facts"]):
                rating = "Supported"#check_atomic_fact(fact, bio_person, llm, max_steps=2)[0].answer
                fact = {"fact": fact, "rating": rating}
                sentence_facts["atomic_facts"][fact_index] = fact
                with pbar_lock:
                    pbar.update(1)
        with results_lock:
            results.append(full_dict)
    except Exception as e:
        print(f"Error processing response {full_dict['id']}: {e}")
        with error_log_lock:
            error_log.append(f"Error processing response {full_dict['id']}: {e}")
    finally:
        request_semaphore.release()

first_n = None

total_facts = len([fact for query in atomic_fact_data for sentence_facts in query["results"]["all_atomic_facts"] for fact in sentence_facts["atomic_facts"]])

threads = []
# Fix: enumerate returns (index, item), so iterate directly
with tqdm(total=total_facts, desc="Proce    ssing responses") as pbar:
    for query in atomic_fact_data[:first_n]:
        t = threading.Thread(target=worker, args=(query, llm, pbar))
        t.start()
        threads.append(t)

    for t in threads:
        t.join()

with open(os.getcwd() + "/data_for_git/fact_rating_log.txt", "w") as f:
    for error in error_log:
        f.write(error + "\n")

# Now results contains all the atomic facts
print(f"Processed {len(results)} responses")
# Access results: results[0], results[1], etc.

Proce    ssing responses: 100%|██████████| 172/172 [00:00<00:00, 40246.60it/s]

Processed 25 responses


In [ ]:
#write results to file
with open(os.getcwd() + "/data_for_git/rated_facts.jsonl", "w") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")


: 